[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 04](README.md)

# MPI punto a punto y progreso

**Tema:** 04 · **Sesiones:** 16, 17, 18 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo hacer coincidir mensajes sin depender del orden accidental ni introducir interbloqueos?


## Resultados de aprendizaje

- Identificar comunicador, rango, tag, datatype y count.
- Comparar bloqueo, no bloqueo y `MPI_Sendrecv`.
- Validar emparejamiento y vida útil de buffers.


## Modelo conceptual

Un mensaje coincide por comunicador, origen permitido y tag; datatype/count determinan interpretación y capacidad.

`MPI_Isend/Irecv` inicia operaciones cuyos buffers no pueden reutilizarse hasta completar la solicitud.

El progreso y el buffering no deben usarse para justificar un patrón potencialmente bloqueante.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "04"
NOTEBOOK = "04_mpi/01_punto_a_punto.ipynb"
assert (ROOT / "curso" / "notebooks" / "04_mpi" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Anillo determinista

Se genera el contrato de mensajes para cada rango.


In [ ]:
def ring_contract(size, tag=17):
    return [{"rank": r, "send_to": (r+1)%size, "recv_from": (r-1)%size, "tag": tag} for r in range(size)]
contract = ring_contract(6)
for row in contract:
    sender = row["recv_from"]
    assert (sender + 1) % 6 == row["rank"]
    print(row)


**Interpretación.** El mismo tag es seguro dentro del contrato del anillo; fases distintas deben distinguirse o sincronizarse.


## Descomposición de halo

Se calculan vecinos y rangos de un dominio 1D, incluidos extremos físicos.


In [ ]:
n, size = 25, 4
q, r = divmod(n, size)
start = 0
rows = []
for rank in range(size):
    local = q + (rank < r)
    rows.append((rank, start, start+local, rank-1 if rank else None, rank+1 if rank+1<size else None))
    start += local
assert rows[-1][2] == n
for row in rows: print(row)


**Interpretación.** Los procesos extremos usan condiciones de frontera o `MPI_PROC_NULL`; no reciben un halo inexistente.


## Práctica reproducible

1. Compilar `mpi/hello_mpi.c` y `mpi/ring_pass.c`.
2. Construir una variante `Sendrecv` y otra no bloqueante.
3. Probar tamaños de proceso 1, 2 y mayores que dos.


## Errores frecuentes

- Asumir que `MPI_Send` siempre bufferiza.
- Reutilizar un buffer antes de Wait.
- Ignorar status y tamaño recibido.

## Criterios de aceptación

- Todos los mensajes tienen contrato compatible.
- No hay deadlock para tamaños admitidos.
- Errores MPI y códigos de salida se comprueban.


## Referencias y material relacionado

- [Hello MPI](../../../mpi/hello_mpi.c)
- [Anillo](../../../mpi/ring_pass.c)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 04](README.md)
